# 🗺️ Notebook 3 — Gerador de Dados Geoespaciais: Setor de Energia
### Workshop Databricks para Times de Negócio | Módulo de Geoprocessamento

Este notebook gera um dataset **geoespacial realístico** de uma distribuidora de energia que atua em **São Paulo (SP)** e **Piracicaba (SP)**. Todos os pontos, polígonos e linhas ficam **dentro dos limites dessas duas cidades**.

As geometrias são armazenadas como **WKT (Well-Known Text)** em uma coluna `geometria_wkt`, além das colunas `latitude`/`longitude` — pronto para ser usado com as funções espaciais `ST_*` e `H3` do Databricks, com mapas no AI/BI e com perguntas no Genie.

| Tabela | Geometria | Descrição |
|---|---|---|
| **clientes_geo** | `POINT` | Localização (lat/long) dos 200 clientes, ligada por `cliente_id` às tabelas do Notebook 1 |
| **subestacoes** | `POINT` | Subestações de energia com capacidade (MVA) |
| **areas_atendimento** | `POLYGON` | Zonas/regiões de atendimento (para choropleth e `ST_Contains`) |
| **ocorrencias** | `POINT` | Interrupções/faltas de energia com data, tipo e clientes afetados |
| **rede_distribuicao** | `LINESTRING` | Linhas de distribuição ligando subestações às zonas |

Os dados são salvos como **tabelas Delta no Unity Catalog** e como **arquivos CSV em um Volume**.

> 💡 **Pré-requisito:** rode o **Notebook 1** antes, usando o mesmo `catálogo`, `schema` e `prefixo`. A tabela `clientes_geo` usa os mesmos `cliente_id` (1 a 200), permitindo joins com `consumo` e `instalacoes`.

## 📋 Descrição das Colunas — Tabelas Geoespaciais

### Tabela `clientes_geo` (POINT)
| Coluna | Descrição |
|---|---|
| cliente_id | Identificador do cliente (mesmo do Notebook 1: 1 a 200) |
| cidade | Cidade do cliente (São Paulo ou Piracicaba) |
| bairro | Bairro / região do cliente |
| latitude | Latitude do ponto de entrega |
| longitude | Longitude do ponto de entrega |
| geometria_wkt | Geometria `POINT(long lat)` em formato WKT |

### Tabela `subestacoes` (POINT)
| Coluna | Descrição |
|---|---|
| subestacao_id | Identificador da subestação |
| nome | Nome da subestação |
| cidade | Cidade |
| bairro | Bairro / região |
| latitude / longitude | Coordenadas da subestação |
| capacidade_mva | Capacidade instalada em MVA |
| geometria_wkt | Geometria `POINT(long lat)` em WKT |

### Tabela `areas_atendimento` (POLYGON)
| Coluna | Descrição |
|---|---|
| area_id | Identificador da zona |
| nome_area | Nome da zona de atendimento |
| cidade | Cidade |
| populacao | População estimada atendida |
| geometria_wkt | Geometria `POLYGON` (fronteira da zona) em WKT |

### Tabela `ocorrencias` (POINT)
| Coluna | Descrição |
|---|---|
| ocorrencia_id | Identificador da ocorrência |
| cliente_id | Cliente associado (referência a clientes_geo) |
| cidade / bairro | Localização |
| latitude / longitude | Coordenadas da ocorrência |
| data_hora | Data e hora do evento |
| tipo | Tipo da ocorrência (queda de linha, curto, etc.) |
| clientes_afetados | Nº de clientes afetados |
| duracao_min | Duração da interrupção (minutos) |
| geometria_wkt | Geometria `POINT(long lat)` em WKT |

### Tabela `rede_distribuicao` (LINESTRING)
| Coluna | Descrição |
|---|---|
| linha_id | Identificador da linha |
| subestacao_id | Subestação de origem |
| cidade | Cidade |
| zona_destino | Zona de atendimento de destino |
| tensao_kv | Tensão da linha (kV) |
| comprimento_km | Comprimento aproximado (km) |
| geometria_wkt | Geometria `LINESTRING` (traçado) em WKT |

## ⚙️ Passo 1 — Configurar Parâmetros (Widgets)

Preencha os widgets abaixo com as informações do seu ambiente (use os **mesmos** valores do Notebook 1):
- **nome_catalogo**: Catálogo do Unity Catalog onde as tabelas serão criadas
- **nome_schema**: Schema (banco de dados) onde as tabelas serão criadas
- **seu_prefixo**: Prefixo único para evitar conflitos entre participantes (ex: `carol_`)
- **caminho_volume**: Caminho do Volume onde os CSVs serão salvos

In [ ]:
dbutils.widgets.removeAll()

In [ ]:
dbutils.widgets.removeAll()

dbutils.widgets.text("nome_catalogo","","Catálogo")
dbutils.widgets.text("nome_schema","","Schema")
dbutils.widgets.text("seu_prefixo","","Seu Prefixo")
dbutils.widgets.text("caminho_volume", "", "Caminho do Volume")

In [ ]:
nome_catalogo  = dbutils.widgets.get("nome_catalogo")
nome_schema    = dbutils.widgets.get("nome_schema")
seu_prefixo    = dbutils.widgets.get("seu_prefixo")
caminho_volume = dbutils.widgets.get("caminho_volume")

print(f"Catálogo : {nome_catalogo}")
print(f"Schema   : {nome_schema}")
print(f"Prefixo  : {seu_prefixo}")
print(f"Volume   : {caminho_volume}")

## 📦 Passo 2 — Importações

In [ ]:
import random
import datetime
import math
import pandas as pd
import numpy as np
from pyspark.sql.types import *
import os

random.seed(42)
np.random.seed(42)

## 🗄️ Passo 3 — Criar Schema e Volume no Unity Catalog

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {nome_catalogo}.{nome_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {nome_catalogo}.{nome_schema}.arquivos")

print(f"✅ Schema : {nome_catalogo}.{nome_schema}")
print(f"✅ Volume : {nome_catalogo}.{nome_schema}.arquivos")

## 🧭 Passo 4 — Referências Geográficas e Helpers de WKT

Definimos os **centros reais dos bairros** de São Paulo e Piracicaba, além de funções auxiliares para montar geometrias em **WKT**.

> ⚠️ **Importante sobre WKT:** a ordem das coordenadas é `X Y`, ou seja, **`LONGITUDE LATITUDE`** (e não latitude/longitude). As funções abaixo já cuidam disso.

In [ ]:
# ------------------------------------------------------------------
# Centros REAIS dos bairros — tudo dentro de São Paulo (SP) e
# Piracicaba (SP). Coordenadas em (latitude, longitude).
# ------------------------------------------------------------------
BAIRROS = {
    "São Paulo": {
        "Sé":           (-23.5505, -46.6333),
        "Pinheiros":    (-23.5670, -46.7020),
        "Vila Mariana": (-23.5890, -46.6340),
        "Santana":      (-23.5020, -46.6250),
        "Itaquera":     (-23.5400, -46.4560),
        "Mooca":        (-23.5580, -46.6000),
        "Butantã":      (-23.5710, -46.7080),
        "Tatuapé":      (-23.5400, -46.5760),
        "Santo Amaro":  (-23.6540, -46.7090),
        "Lapa":         (-23.5200, -46.7050),
    },
    "Piracicaba": {
        "Centro":          (-22.7250, -47.6490),
        "Paulista":        (-22.7060, -47.6340),
        "São Dimas":       (-22.7150, -47.6600),
        "Vila Rezende":    (-22.7000, -47.6550),
        "Cidade Alta":     (-22.7350, -47.6400),
        "Santa Terezinha": (-22.7400, -47.6600),
    },
}

# ------------------------------------------------------------------
# Helpers de WKT — lembrando: ordem é LONGITUDE LATITUDE (X Y).
# ------------------------------------------------------------------
def wkt_point(lon, lat):
    return f"POINT({lon:.6f} {lat:.6f})"

def wkt_polygon(coords):
    # coords: lista de (lon, lat); o anel deve ser fechado (1o = ultimo)
    anel = ", ".join(f"{lon:.6f} {lat:.6f}" for lon, lat in coords)
    return f"POLYGON(({anel}))"

def wkt_linestring(coords):
    seg = ", ".join(f"{lon:.6f} {lat:.6f}" for lon, lat in coords)
    return f"LINESTRING({seg})"

def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi   = math.radians(lat2 - lat1)
    dlmb   = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlmb/2)**2
    return round(2 * R * math.asin(math.sqrt(a)), 3)

def retangulo(lat0, lon0, meia=0.030):
    # devolve os vertices (lon, lat) de um POLYGON retangular ao redor de um centro
    return [
        (lon0 - meia, lat0 - meia),
        (lon0 + meia, lat0 - meia),
        (lon0 + meia, lat0 + meia),
        (lon0 - meia, lat0 + meia),
        (lon0 - meia, lat0 - meia),
    ]

print("✅ Referências geográficas e helpers de WKT carregados.")

### Tabela 1: `clientes_geo` (POINT)
Georreferencia os **200 clientes** (mesmos `cliente_id` do Notebook 1) em bairros de São Paulo e Piracicaba. Permite join com `consumo` e `instalacoes`.

In [ ]:
NUM_CLIENTES = 200  # mesmos cliente_id do Notebook 1 (1..200)

clientes_geo = []
for cid in range(1, NUM_CLIENTES + 1):
    cidade = random.choices(["São Paulo", "Piracicaba"], weights=[70, 30])[0]
    bairro = random.choice(list(BAIRROS[cidade].keys()))
    lat0, lon0 = BAIRROS[cidade][bairro]

    # jitter de ~ ±1.3 km para espalhar os pontos dentro do bairro
    lat = round(lat0 + random.uniform(-0.012, 0.012), 6)
    lon = round(lon0 + random.uniform(-0.012, 0.012), 6)

    clientes_geo.append({
        "cliente_id":    cid,
        "cidade":        cidade,
        "bairro":        bairro,
        "latitude":      lat,
        "longitude":     lon,
        "geometria_wkt": wkt_point(lon, lat),
    })

df_clientes_geo = pd.DataFrame(clientes_geo)
print(f"Clientes georreferenciados: {len(df_clientes_geo)}")
print(df_clientes_geo["cidade"].value_counts().to_string())
display(df_clientes_geo.head(5))

### Tabela 2: `subestacoes` (POINT)
Subestações de energia em São Paulo e Piracicaba, com capacidade instalada (MVA). Servem de referência para cálculos de distância (`ST_Distance`) até os clientes.

In [ ]:
SUBESTACOES = [
    # (nome, cidade, bairro, lat, lon, capacidade_mva)
    ("SE Sé",                    "São Paulo",  "Sé",              -23.5505, -46.6333, 300),
    ("SE Santana",               "São Paulo",  "Santana",         -23.5020, -46.6250, 250),
    ("SE Itaquera",              "São Paulo",  "Itaquera",        -23.5400, -46.4560, 200),
    ("SE Butantã",               "São Paulo",  "Butantã",         -23.5710, -46.7080, 220),
    ("SE Santo Amaro",           "São Paulo",  "Santo Amaro",     -23.6540, -46.7090, 280),
    ("SE Piracicaba Centro",     "Piracicaba", "Centro",          -22.7250, -47.6490, 120),
    ("SE Piracicaba Industrial", "Piracicaba", "Santa Terezinha", -22.7400, -47.6600, 150),
]

subestacoes = []
for i, (nome, cidade, bairro, lat, lon, cap) in enumerate(SUBESTACOES, start=1):
    subestacoes.append({
        "subestacao_id":  i,
        "nome":           nome,
        "cidade":         cidade,
        "bairro":         bairro,
        "latitude":       lat,
        "longitude":      lon,
        "capacidade_mva": cap,
        "geometria_wkt":  wkt_point(lon, lat),
    })

df_subestacoes = pd.DataFrame(subestacoes)
print(f"Subestações geradas: {len(df_subestacoes)}")
display(df_subestacoes)

### Tabela 3: `areas_atendimento` (POLYGON)
Zonas de atendimento como **polígonos**. São a base para o **mapa coroplético (choropleth)** no AI/BI e para consultas do tipo "qual cliente está dentro de qual zona" (`ST_Contains`).

In [ ]:
ZONAS = [
    # (nome_area, cidade, lat_centro, lon_centro, meia_dimensao)
    ("SP - Centro",     "São Paulo",  -23.5505, -46.6333, 0.030),
    ("SP - Zona Norte", "São Paulo",  -23.5020, -46.6250, 0.035),
    ("SP - Zona Sul",   "São Paulo",  -23.6540, -46.7090, 0.040),
    ("SP - Zona Leste", "São Paulo",  -23.5400, -46.4900, 0.045),
    ("SP - Zona Oeste", "São Paulo",  -23.5690, -46.7050, 0.035),
    ("PIRA - Norte",    "Piracicaba", -22.7050, -47.6450, 0.030),
    ("PIRA - Sul",      "Piracicaba", -22.7400, -47.6550, 0.030),
]

areas = []
for i, (nome, cidade, lat0, lon0, meia) in enumerate(ZONAS, start=1):
    areas.append({
        "area_id":       i,
        "nome_area":     nome,
        "cidade":        cidade,
        "populacao":     random.randint(80_000, 900_000),
        "geometria_wkt": wkt_polygon(retangulo(lat0, lon0, meia)),
    })

df_areas = pd.DataFrame(areas)
print(f"Áreas de atendimento (polígonos) geradas: {len(df_areas)}")
display(df_areas)

### Tabela 4: `ocorrencias` (POINT)
Interrupções/faltas de energia com **espaço + tempo + severidade**. Ótima para mapas de calor com **H3**, análise de clusters de falha e `ST_Buffer` (clientes afetados num raio).

In [ ]:
TIPOS_OCORRENCIA = ["Queda de Linha", "Curto-Circuito", "Sobrecarga",
                    "Falha em Transformador", "Poda de Árvore", "Descarga Atmosférica"]

NUM_OCORRENCIAS = 300
ocorrencias = []
DATA_INI = datetime.datetime(2024, 1, 1)

for oid in range(1, NUM_OCORRENCIAS + 1):
    cidade = random.choices(["São Paulo", "Piracicaba"], weights=[70, 30])[0]
    bairro = random.choice(list(BAIRROS[cidade].keys()))
    lat0, lon0 = BAIRROS[cidade][bairro]

    lat = round(lat0 + random.uniform(-0.015, 0.015), 6)
    lon = round(lon0 + random.uniform(-0.015, 0.015), 6)

    data_hora = DATA_INI + datetime.timedelta(
        days=random.randint(0, 700), minutes=random.randint(0, 1440)
    )

    ocorrencias.append({
        "ocorrencia_id":     oid,
        "cliente_id":        random.randint(1, NUM_CLIENTES),
        "cidade":            cidade,
        "bairro":            bairro,
        "latitude":          lat,
        "longitude":         lon,
        "data_hora":         data_hora.strftime("%Y-%m-%d %H:%M:%S"),
        "tipo":              random.choice(TIPOS_OCORRENCIA),
        "clientes_afetados": random.randint(1, 5000),
        "duracao_min":       random.randint(5, 480),
        "geometria_wkt":     wkt_point(lon, lat),
    })

df_ocorrencias = pd.DataFrame(ocorrencias)
print(f"Ocorrências geradas: {len(df_ocorrencias)}")
display(df_ocorrencias.head(5))

### Tabela 5: `rede_distribuicao` (LINESTRING)
Linhas de distribuição ligando cada subestação às zonas de atendimento. Habilita `ST_Length` (comprimento) e `ST_Intersects` (quais ocorrências caem sobre quais linhas).

In [ ]:
# Cada subestação alimenta até 2 zonas da MESMA cidade -> linhas de distribuição
centros_zona = {z[0]: (z[2], z[3], z[1]) for z in ZONAS}  # nome -> (lat, lon, cidade)

rede = []
linha_id = 1
for sub in subestacoes:
    zonas_cidade = [nome for nome, (la, lo, cid) in centros_zona.items()
                    if cid == sub["cidade"]]
    for nome_zona in random.sample(zonas_cidade, k=min(2, len(zonas_cidade))):
        lat_z, lon_z, _ = centros_zona[nome_zona]
        comp = haversine_km(sub["longitude"], sub["latitude"], lon_z, lat_z)
        rede.append({
            "linha_id":       linha_id,
            "subestacao_id":  sub["subestacao_id"],
            "cidade":         sub["cidade"],
            "zona_destino":   nome_zona,
            "tensao_kv":      random.choice([13.8, 34.5, 69.0, 138.0]),
            "comprimento_km": comp,
            "geometria_wkt":  wkt_linestring([
                                  (sub["longitude"], sub["latitude"]),
                                  (lon_z, lat_z),
                              ]),
        })
        linha_id += 1

df_rede = pd.DataFrame(rede)
print(f"Linhas de distribuição geradas: {len(df_rede)}")
display(df_rede.head(5))

## 💾 Passo 5 — Salvar Tabelas no Unity Catalog

In [ ]:
tabelas = {
    "clientes_geo":      df_clientes_geo,
    "subestacoes":       df_subestacoes,
    "areas_atendimento": df_areas,
    "ocorrencias":       df_ocorrencias,
    "rede_distribuicao": df_rede,
}

for nome_tabela, df_pandas in tabelas.items():
    nome_completo = f"{nome_catalogo}.{nome_schema}.{seu_prefixo}_{nome_tabela}"
    df_spark = spark.createDataFrame(df_pandas)
    (
        df_spark.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_completo)
    )
    print(f"✅ Tabela salva: {nome_completo}  ({df_pandas.shape[0]} linhas, {df_pandas.shape[1]} colunas)")

## 📁 Passo 6 — Salvar CSVs no Volume

In [ ]:
import shutil

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = '/Workspace' + os.path.dirname(notebook_path)

for nome_tabela, df_pandas in tabelas.items():
    caminho_csv = f"/{notebook_dir}/{seu_prefixo}_{nome_tabela}.csv"
    df_pandas.to_csv(caminho_csv, index=False, encoding="utf-8")

for nome_tabela in tabelas.keys():
    origem  = f"/{notebook_dir}/{seu_prefixo}_{nome_tabela}.csv"
    destino = f"{caminho_volume}{seu_prefixo}_{nome_tabela}.csv"
    shutil.copy(origem, destino)
    print(f"✅ Arquivo copiado para o volume: {destino}")

## ✅ Passo 7 — Resumo Final

In [ ]:
print("=" * 70)
print("  RESUMO — DATASET GEOESPACIAL (São Paulo + Piracicaba) GERADO")
print("=" * 70)

for nome_tabela, df_pandas in tabelas.items():
    nome_completo = f"{nome_catalogo}.{nome_schema}.{seu_prefixo}_{nome_tabela}"
    print(f"\n🗺️  {nome_tabela.upper()}")
    print(f"   Delta Table : {nome_completo}")
    print(f"   CSV         : {caminho_volume}{seu_prefixo}_{nome_tabela}.csv")
    print(f"   Linhas      : {df_pandas.shape[0]}")
    print(f"   Colunas     : {', '.join(df_pandas.columns.tolist())}")

print("\n" + "=" * 70)
print("🎉 Pronto! Geometrias em WKT prontas para ST_*, H3, Genie e AI/BI.")
print("   Próximo passo: abra o Notebook 2 → Módulo 8 (Geoprocessamento).")
print("=" * 70)